In [1]:
!pip install arxiv

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/81.5 kB ? eta -:--:--
   ---------------------------------------- 81.5/81.5 kB 2.3 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6104 sha256=e23c8a6a3893d898731d4ca23271e5d7c0402d4e335c88a2b38652f68b69cab2
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\3b\25\2a\105d6a15df6914f4d15047691c6c28f9052cc1173e40285d03
Successfully built sgmllib3k



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
from dotenv import load_dotenv
import os

load_dotenv() 

True

In [4]:
from langchain_community.tools  import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [5]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
tool=WikipediaQueryRun(api_wrapper=api_wrapper)

In [7]:
tool.name

'wikipedia'

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface.llms import HuggingFaceEndpoint
from langchain_huggingface.chat_models import ChatHuggingFace

c:\Users\DELL\OneDrive\Desktop\LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [58]:
from langchain_groq import ChatGroq
import os

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

In [12]:
loader=WebBaseLoader("https://www.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=20).split_documents(docs)
db=FAISS.from_documents(documents,embeddings)
retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000018F934AAFD0>, search_kwargs={})

In [22]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever=retriever, name="LangChain_Retriever", description="useful for when you need to answer questions about LangChain website")

In [ ]:
# retriever_tool.name

'LangChain_Retriever'

In [24]:
# Arxiv tool
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

arxiv_api=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_api)
arxiv.name

'arxiv'

In [25]:
tools=[tool,retriever_tool,arxiv]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\DELL\\OneDrive\\Desktop\\LangChain\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='LangChain_Retriever', description='useful for when you need to answer questions about LangChain website', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000018F97651760>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000018F97778D60>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200))]

In [59]:
from langchain.agents import create_agent
agent = create_agent(
    model,
    tools=tools
)


In [42]:
user_query='tell me about langchain and also tell me about the latest research papers in langchain'

In [60]:
response=agent.invoke({
    "messages":[{"role":"user","content":user_query}]
})

In [63]:
response

{'messages': [HumanMessage(content='tell me about langchain and also tell me about the latest research papers in langchain', additional_kwargs={}, response_metadata={}, id='60c35bd7-81a4-460c-8e3d-40d77bb6eb1e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5rh6r8daq', 'function': {'arguments': '{"query":"LangChain overview and latest research papers"}', 'name': 'LangChain_Retriever'}, 'type': 'function'}, {'id': '213pxrt4w', 'function': {'arguments': '{"query":"LangChain latest research papers"}', 'name': 'arxiv'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 528, 'total_tokens': 570, 'completion_time': 0.081616672, 'completion_tokens_details': None, 'prompt_time': 0.025746186, 'prompt_tokens_details': None, 'queue_time': 0.16210944, 'total_time': 0.107362858}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs':